# 01 · EDA and Data Generation

What the synthetic universe actually looks like, and whether the two properties the rest of
the system depends on are really there:

1. **A fat-headed device distribution** — one stock user-agent colliding across ~1,488
   accounts, which is what forces collision capping to exist.
2. **Five disjoint worlds** — no card in two worlds, so leave-one-world-out validation is
   an honest generalisation test rather than a leak.

Run `make all` first so the artifacts exist.

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False})
print("root:", ROOT)

In [ ]:
from src.data_generator import load_ground_truth, load_transactions
from src.config import CARDS_CSV, MERCHANTS_CSV, GENERATION

txns = load_transactions()
cards = pd.read_csv(CARDS_CSV)
merchants = pd.read_csv(MERCHANTS_CSV)
rings = load_ground_truth()

print(f"{len(txns):,} transactions | {len(cards):,} cards | {len(merchants):,} merchants | {len(rings)} rings")
txns.head(3)

## 1 · Amount distribution — why probes are not trivially separable

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

ax[0].hist(np.log10(txns["amount"].clip(lower=1)), bins=80, color="#64748b")
ax[0].axvline(1, color="#38bdf8", ls="--", label="probe ceiling (INR 10)")
ax[0].axvline(4, color="#f87171", ls="--", label="cashout floor (INR 10,000)")
ax[0].set_xlabel("log10(amount, INR)"); ax[0].set_ylabel("transactions")
ax[0].set_title("All transactions"); ax[0].legend()

for role, colour in [("legit", "#64748b"), ("probe", "#38bdf8"), ("cashout", "#f87171")]:
    sub = txns.loc[txns["txn_role"] == role, "amount"]
    if len(sub):
        ax[1].hist(np.log10(sub.clip(lower=1)), bins=60, alpha=0.65, label=f"{role} (n={len(sub):,})", color=colour)
ax[1].set_xlabel("log10(amount, INR)"); ax[1].set_title("By transaction role"); ax[1].legend()
plt.tight_layout()

overlap = ((txns["txn_role"] == "legit") & (txns["amount"] < 10)).sum()
print(f"Legitimate transactions under INR 10: {overlap:,} "
      f"({100 * overlap / len(txns):.2f}% of all traffic)")
print("Amounts are drawn from each MERCHANT's own ticket distribution, not a global one.")
print("A INR 5 charge at a recharge merchant is genuinely unremarkable -- which is exactly")
print("why probe_fraction only becomes discriminative combined with cross-merchant structure.")

## 2 · The identity layer — the fat head that forces capping

Real device fingerprinting has a fat head: one stock user-agent covers a large slice of the
population. Build an identity graph naively and that single value fuses everything into one
component.

In [ ]:
device_card = txns.groupby("device_fingerprint")["card_id"].nunique()
ip_card = txns.groupby("ip_address")["card_id"].nunique()
email_card = txns.groupby("email_hash")["card_id"].nunique()

summary = pd.DataFrame({
    "device_fingerprint": device_card.describe(percentiles=[.5, .95, .99]),
    "ip_address": ip_card.describe(percentiles=[.5, .95, .99]),
    "email_hash": email_card.describe(percentiles=[.5, .95, .99]),
}).T[["count", "50%", "95%", "99%", "max"]]
summary.columns = ["n_values", "p50", "p95", "p99", "max"]
display(summary)

generic = device_card.get(GENERATION.generic_device_string, 0)
print(f"\nGeneric user-agent collides across {generic:,} accounts.")
print(f"Uncapped, that single value alone expands to {generic * (generic - 1) // 2:,} edges.")
print(f"Capped at 12, it expands to {12 * 11 // 2} edges.")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(device_card.values, bins=np.logspace(0, np.log10(device_card.max() + 1), 50), color="#a78bfa")
ax.set_xscale("log"); ax.set_yscale("log")
ax.axvline(4, color="#4ade80", ls="--", label="raw p95 = 4 cards")
ax.axvline(12, color="#fbbf24", ls="-", lw=2, label="cap applied = 12 (floored)")
ax.axvline(generic, color="#f87171", ls="--", label=f"generic UA = {generic:,} cards")
ax.set_xlabel("cards sharing one device fingerprint"); ax.set_ylabel("number of device values")
ax.set_title("Device cardinality is fat-headed -- the cap exists for the right tail")
ax.legend(); plt.tight_layout()

print("The cap FLOOR matters as much as the cap. The raw p95 is 4 cards; capping there")
print(f"would shred the {GENERATION.ring_size_max}-card rings we exist to find. The floor is")
print("set at 12 -- above max ring size -- so the cap only ever bites far outside ring scale.")

## 3 · Why there are no merchant edges

In [ ]:
from src.graph_builder import get_collision_stats

stats = get_collision_stats(txns)
m = stats["merchant"]
print(f"Total possible card pairs        : {m['total_possible_card_pairs']:,}")
print(f"Pairs sharing a MERCHANT         : {m['card_pairs_sharing_a_merchant']:,} "
      f"({m['merchant_pair_saturation']:.1%})")
print(f"Pairs sharing an IDENTITY value  : {m['card_pairs_sharing_an_identity']:,} "
      f"({m['identity_pair_saturation']:.3%})")
print()
print(m["verdict"])

## 4 · World disjointness — the premise of cross-world validation

In [ ]:
print("Cards per world:"); print(cards["world_id"].value_counts().sort_index().to_string())
assert cards.groupby("card_id")["world_id"].nunique().max() == 1, "a card spans two worlds"

world_of = dict(zip(cards["card_id"], cards["world_id"]))
straddling = sum(1 for r in rings.itertuples(index=False)
                 if len({world_of[c] for c in r.card_id_list}) > 1)
print(f"\nRings straddling a world boundary: {straddling}  (must be 0)")
assert straddling == 0
print("Leave-one-world-out validation is therefore a genuine generalisation test:")
print("every test candidate comes from a population the model has never seen a card from.")

## 5 · The ring signature vs the hard negatives

The interesting comparison is not ring-vs-random. It is **ring vs household burst** — a
family sharing one tablet, buying four recharges in three minutes and then a INR 6,000
appliance an hour later. Same device, same compressed timing, same escalation direction.
Two of the four ring signatures fire.

If those did not exist in the data, the model would separate rings on structure alone and
the reported precision would be meaningless.

In [ ]:
groups = {
    "ring (probe+cashout)": txns[txns["txn_role"].isin(["probe", "cashout"])],
    "household burst": txns[txns["txn_role"].str.startswith("household_burst")],
    "ordinary legit": txns[txns["txn_role"] == "legit"].sample(20000, random_state=42),
}
rows = []
for name, g in groups.items():
    rows.append({
        "population": name, "n_txns": len(g),
        "% under INR 10": f"{(g['amount'] < 10).mean():.1%}",
        "% over INR 10k": f"{(g['amount'] > 10_000).mean():.1%}",
        "median amount": f"{g['amount'].median():,.0f}",
        "max/min ratio": f"{g['amount'].max() / max(g['amount'].min(), 0.01):,.0f}x",
        "mean vulcan": f"{g['vulcan_score'].mean():.3f}",
    })
display(pd.DataFrame(rows).set_index("population"))

print("Household bursts fire probe-like small amounts and compressed timing, but their")
print("escalation is ~100x against a ring's ~5,000x and their cashout fraction is ~0.")
print("That gap is what the behavioural features have to find. It is not a free win.")

## 6 · Vulcan scores — the gap this system exists to close

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for role, colour, label in [("legit", "#64748b", "legit"),
                            ("probe", "#38bdf8", "ring probe"),
                            ("cashout", "#f87171", "ring cashout")]:
    sub = txns.loc[txns["txn_role"] == role, "vulcan_score"]
    if len(sub):
        ax.hist(sub, bins=50, alpha=0.65, density=True, label=f"{label} (mean {sub.mean():.2f})", color=colour)
ax.axvline(0.4, color="#fbbf24", ls="--", label="REVIEW band opens (0.4)")
ax.axvline(0.7, color="#f87171", ls="--", label="BLOCK band opens (0.7)")
ax.set_xlabel("simulated Vulcan per-transaction score"); ax.set_ylabel("density")
ax.set_title("Every ring transaction is individually plausible to a per-transaction model")
ax.legend(); plt.tight_layout()

cash = txns.loc[txns["txn_role"] == "cashout", "vulcan_score"]
print(f"Ring cashouts scoring above the BLOCK band on Vulcan alone: {(cash >= 0.7).mean():.1%}")
print("That is the whole argument. Vulcan is not wrong -- the evidence is not in the")
print("transaction. It is in the space between them, which is what Ringfence supplies.")